# Piper-Plus ファインチューニング (Colab)

6言語マルチリンガルベースモデル (571話者, 75 epoch) から、あなたの音声データを使ったシングルスピーカーファインチューニングを行います。

**ベースモデル:** `ayousanz/piper-plus-base` (6lang, epoch=74-step=504712)

**必要なデータ:** LJSpeech 形式のデータセット
- `wavs/` フォルダ (WAV ファイル)
- `metadata.csv` (パイプ区切り: `ファイル名|テキスト|テキスト`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayutaz/piper-plus/blob/dev/notebooks/finetune.ipynb)

## 0. GPU 確認

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. 環境セットアップ

In [ ]:
# piper-plus リポジトリをクローン
!git clone https://github.com/ayutaz/piper-plus.git /content/piper-plus
%cd /content/piper-plus

In [ ]:
# uv インストール + 依存関係セットアップ
!pip install uv
!uv sync

In [ ]:
# 追加依存 (音声処理用)
!uv pip install soxr soundfile

## 2. ベースモデル (6lang チェックポイント) ダウンロード

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download
import os

BASE_MODEL_REPO = "ayousanz/piper-plus-base"
CHECKPOINT_FILENAME = "epoch=74-step=504712.ckpt"

os.makedirs("/content/base-model", exist_ok=True)

ckpt_path = hf_hub_download(
    repo_id=BASE_MODEL_REPO,
    filename=CHECKPOINT_FILENAME,
    local_dir="/content/base-model",
)
print(f"Checkpoint downloaded: {ckpt_path}")

BASE_CHECKPOINT = f"/content/base-model/{CHECKPOINT_FILENAME}"
print(f"Size: {os.path.getsize(BASE_CHECKPOINT) / 1024**2:.1f} MB")

## 3. LJSpeech 形式データセットのアップロード

LJSpeech 形式のデータセットを用意してください:

```
your-dataset/
├── wavs/
│   ├── audio001.wav
│   ├── audio002.wav
│   └── ...
└── metadata.csv
```

**metadata.csv の形式** (パイプ `|` 区切り、ヘッダーなし):
```
audio001|こんにちは、今日は良い天気ですね。|こんにちは、今日は良い天気ですね。
audio002|吾輩は猫である。|吾輩は猫である。
```

アップロード方法は「ZIP アップロード」または「Google Drive」から選択できます。

In [ ]:
# === 設定 ===
# データセットの配置方法を選択してください:
#   "upload"  : ZIP ファイルをアップロード
#   "gdrive"  : Google Drive からコピー
SOURCE_METHOD = "upload"  # "upload" or "gdrive"

# Google Drive 使用時のパス (SOURCE_METHOD="gdrive" の場合)
GDRIVE_DATASET_PATH = "/content/drive/MyDrive/my-dataset.zip"

# 話者名 (モデル識別用)
SPEAKER_NAME = "my_speaker"

# 主要言語 (データセットの言語)
# ja, en, zh, es, fr, pt から選択
PRIMARY_LANGUAGE = "ja"

# LJSpeech データセットの展開先
LJSPEECH_DIR = "/content/ljspeech-dataset"

In [ ]:
import os
import zipfile
import glob
import csv

os.makedirs(LJSPEECH_DIR, exist_ok=True)

if SOURCE_METHOD == "gdrive":
    from google.colab import drive
    drive.mount("/content/drive")
    
    if GDRIVE_DATASET_PATH.endswith(".zip"):
        with zipfile.ZipFile(GDRIVE_DATASET_PATH, "r") as zf:
            zf.extractall(LJSPEECH_DIR)
        print(f"Extracted to {LJSPEECH_DIR}")
    else:
        import shutil
        shutil.copytree(GDRIVE_DATASET_PATH, LJSPEECH_DIR, dirs_exist_ok=True)
        print(f"Copied to {LJSPEECH_DIR}")

elif SOURCE_METHOD == "upload":
    from google.colab import files
    print("LJSpeech 形式の ZIP ファイルをアップロードしてください:")
    print("  (wavs/ フォルダと metadata.csv を含む ZIP)")
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".zip"):
            with zipfile.ZipFile(fname, "r") as zf:
                zf.extractall(LJSPEECH_DIR)
            print(f"Extracted {fname} to {LJSPEECH_DIR}")

# ZIP 内にサブディレクトリがある場合の自動検出
metadata_candidates = glob.glob(f"{LJSPEECH_DIR}/**/metadata.csv", recursive=True)
if metadata_candidates:
    # metadata.csv が見つかったディレクトリを LJSPEECH_DIR に設定
    actual_dir = os.path.dirname(metadata_candidates[0])
    if actual_dir != LJSPEECH_DIR:
        LJSPEECH_DIR = actual_dir
        print(f"Auto-detected dataset directory: {LJSPEECH_DIR}")

# 検証
WAVS_DIR = os.path.join(LJSPEECH_DIR, "wavs")
metadata_path = os.path.join(LJSPEECH_DIR, "metadata.csv")

assert os.path.isdir(WAVS_DIR), f"wavs/ directory not found at {WAVS_DIR}"
assert os.path.isfile(metadata_path), f"metadata.csv not found at {metadata_path}"

wav_files = glob.glob(f"{WAVS_DIR}/*.wav")
with open(metadata_path, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")
    rows = list(reader)

print(f"\nDataset validated:")
print(f"  WAV files: {len(wav_files)}")
print(f"  Metadata entries: {len(rows)}")
if rows:
    print(f"  Sample: {rows[0][0]} | {rows[0][1][:50]}...")

## 4. 音声前処理

WAV ファイルを 22,050Hz にリサンプリングし、音量を正規化します。
既に 22,050Hz/モノラルの場合はスキップされます。

In [ ]:
import os
import numpy as np
import soundfile as sf

try:
    import soxr
    HAS_SOXR = True
except ImportError:
    import torchaudio
    HAS_SOXR = False
    print("soxr not available, using torchaudio for resampling")

TARGET_SR = 22050


def resample_and_normalize(audio_path, output_path, target_sr=TARGET_SR):
    """Resample to target_sr and peak-normalize."""
    audio, src_sr = sf.read(audio_path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)  # stereo -> mono
    
    if src_sr != target_sr:
        if HAS_SOXR:
            audio = soxr.resample(audio, src_sr, target_sr, quality="HQ")
        else:
            import torch
            resampler = torchaudio.transforms.Resample(src_sr, target_sr)
            audio = resampler(torch.from_numpy(audio).unsqueeze(0)).squeeze(0).numpy()
    
    # Peak normalize
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak * 0.95
    
    sf.write(output_path, audio, target_sr, subtype="PCM_16")
    return len(audio) / target_sr


# 全 WAV ファイルをリサンプリング & 正規化
PROCESSED_WAVS_DIR = os.path.join(LJSPEECH_DIR, "wavs_22k")
os.makedirs(PROCESSED_WAVS_DIR, exist_ok=True)

needs_resample = False
# サンプルレート確認
sample_wav = wav_files[0] if wav_files else None
if sample_wav:
    info = sf.info(sample_wav)
    if info.samplerate != TARGET_SR or info.channels > 1:
        needs_resample = True
        print(f"Resampling needed: {info.samplerate}Hz, {info.channels}ch -> {TARGET_SR}Hz, mono")
    else:
        print(f"Audio already at {TARGET_SR}Hz mono, skipping resample")

if needs_resample:
    from tqdm import tqdm
    total_duration = 0
    for wav_path in tqdm(wav_files, desc="Resampling"):
        basename = os.path.basename(wav_path)
        output_path = os.path.join(PROCESSED_WAVS_DIR, basename)
        dur = resample_and_normalize(wav_path, output_path)
        total_duration += dur
    WAVS_DIR = PROCESSED_WAVS_DIR
    print(f"\nResampled {len(wav_files)} files ({total_duration/60:.1f} min total)")
else:
    print(f"Using original wavs: {len(wav_files)} files")

## 5. ファインチューニング用データセット作成

LJSpeech 形式のデータから、Piper 学習用の `dataset.jsonl` + `config.json` を生成します。

In [ ]:
import sys
sys.path.insert(0, "/content/piper-plus/src/python")
sys.path.insert(0, "/content/piper-plus/src/python/g2p")

import json
import csv
import os
import numpy as np
import torch
from pathlib import Path
from hashlib import sha256
from tqdm import tqdm

from piper_plus_g2p.encode.id_maps import get_phoneme_id_map
from piper_train.infer_onnx import text_to_phoneme_ids_and_prosody
from piper_train.vits.mel_processing import spectrogram_torch

# === 設定 ===
DATASET_DIR = f"/content/dataset-{SPEAKER_NAME}-finetune-6lang"
CACHE_DIR = os.path.join(DATASET_DIR, "cache")
SAMPLE_RATE = 22050
LANGUAGE_KEY = "ja-en-zh-es-fr-pt"  # 6lang canonical key

# 言語 ID マッピング (ベースモデルと一致させる)
LANGUAGE_ID_MAP = {"ja": 0, "en": 1, "zh": 2, "es": 3, "fr": 4, "pt": 5}
LANGUAGE_ID = LANGUAGE_ID_MAP[PRIMARY_LANGUAGE]

# Spectrogram parameters (must match training config)
FILTER_LENGTH = 1024
WINDOW_LENGTH = 1024
HOP_LENGTH = 256

os.makedirs(CACHE_DIR, exist_ok=True)

# ID map
id_map = get_phoneme_id_map(LANGUAGE_KEY)
num_symbols = max(max(ids) for ids in id_map.values()) + 1

print(f"Speaker: {SPEAKER_NAME}")
print(f"Primary language: {PRIMARY_LANGUAGE} (id={LANGUAGE_ID})")
print(f"Num symbols: {num_symbols}")

In [ ]:
def encode_text(text, language=PRIMARY_LANGUAGE):
    """Phonemize text and encode to phoneme IDs using the project's standard pipeline."""
    phoneme_ids, prosody_features = text_to_phoneme_ids_and_prosody(
        text,
        phoneme_id_map=id_map,
        language=language,
        language_id_map=LANGUAGE_ID_MAP,
    )
    return phoneme_ids, prosody_features


def cache_audio(wav_path, cache_dir, sample_rate=SAMPLE_RATE):
    """Cache normalized audio and spectrogram."""
    wav_path = Path(wav_path).absolute()
    cache_id = sha256(str(wav_path).encode()).hexdigest()
    norm_path = Path(cache_dir) / f"{cache_id}.pt"
    spec_path = Path(cache_dir) / f"{cache_id}.spec.pt"
    
    if not norm_path.exists():
        audio, sr = sf.read(str(wav_path), dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != sample_rate:
            if HAS_SOXR:
                audio = soxr.resample(audio, sr, sample_rate, quality="HQ")
            else:
                import torchaudio
                resampler = torchaudio.transforms.Resample(sr, sample_rate)
                audio = resampler(torch.from_numpy(audio).unsqueeze(0)).squeeze(0).numpy()
        
        audio_tensor = torch.from_numpy(audio).unsqueeze(0)  # (1, samples)
        torch.save(audio_tensor, str(norm_path))
    else:
        audio_tensor = torch.load(str(norm_path), map_location="cpu", weights_only=False)
    
    if not spec_path.exists():
        spec = spectrogram_torch(
            audio_tensor,
            n_fft=FILTER_LENGTH,
            sampling_rate=sample_rate,
            hop_size=HOP_LENGTH,
            win_size=WINDOW_LENGTH,
            center=False,
        ).squeeze(0)
        torch.save(spec.half(), str(spec_path))
    
    return str(norm_path), str(spec_path)


# テスト (言語に応じたサンプルテキスト)
_test_texts = {
    "ja": "こんにちは、今日は良い天気ですね。",
    "en": "Hello, how are you today?",
    "zh": "你好，今天天气很好。",
    "es": "Hola, ¿cómo estás hoy?",
    "fr": "Bonjour, comment allez-vous?",
    "pt": "Olá, como você está hoje?",
}
test_ids, test_prosody = encode_text(_test_texts.get(PRIMARY_LANGUAGE, _test_texts["en"]))
print(f"Test IDs ({len(test_ids)}): {test_ids[:20]}...")
print(f"Prosody features: {len(test_prosody)} entries")

In [ ]:
# === dataset.jsonl 生成 ===
dataset_jsonl_path = os.path.join(DATASET_DIR, "dataset.jsonl")
utterances = []
errors = []

# metadata.csv を読み込み
with open(metadata_path, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")
    rows = list(reader)

print(f"Processing {len(rows)} utterances...")

for row in tqdm(rows):
    # LJSpeech 形式: id|text|text (3列) または id|text (2列)
    wav_id = row[0]
    text = row[1]
    wav_path = os.path.join(WAVS_DIR, f"{wav_id}.wav")
    if not os.path.exists(wav_path):
        errors.append(f"Missing: {wav_path}")
        continue
    
    try:
        phoneme_ids, prosody_features = encode_text(text, language=PRIMARY_LANGUAGE)
        norm_path, spec_path = cache_audio(wav_path, CACHE_DIR)
        
        prosody_serializable = None
        if prosody_features:
            prosody_serializable = [
                {"a1": p["a1"], "a2": p["a2"], "a3": p["a3"]} if p else None
                for p in prosody_features
            ]
        
        utterance = {
            "text": text,
            "phoneme_ids": phoneme_ids,
            "audio_norm_path": norm_path,
            "audio_spec_path": spec_path,
            "speaker_id": 0,
            "language_id": LANGUAGE_ID,
        }
        if prosody_serializable:
            utterance["prosody_features"] = prosody_serializable
        
        utterances.append(utterance)
    except Exception as e:
        errors.append(f"Error processing {wav_id}: {e}")

with open(dataset_jsonl_path, "w", encoding="utf-8") as f:
    for utt in utterances:
        f.write(json.dumps(utt, ensure_ascii=False) + "\n")

print(f"\nDataset written: {dataset_jsonl_path}")
print(f"Total utterances: {len(utterances)}")
if errors:
    print(f"Errors: {len(errors)}")
    for e in errors[:5]:
        print(f"  {e}")

In [ ]:
# === config.json 生成 ===
config = {
    "audio": {
        "sample_rate": SAMPLE_RATE,
        "filter_length": FILTER_LENGTH,
        "hop_length": HOP_LENGTH,
        "win_length": WINDOW_LENGTH,
    },
    "num_symbols": num_symbols,
    "num_speakers": 1,
    "num_languages": len(LANGUAGE_ID_MAP),
    "language_id_map": LANGUAGE_ID_MAP,
    "speaker_id_map": {SPEAKER_NAME: 0},
    "phoneme_id_map": {k: v for k, v in id_map.items()},
}

config_path = os.path.join(DATASET_DIR, "config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f"Config written: {config_path}")
print(f"  speaker: {SPEAKER_NAME}")
print(f"  num_symbols: {config['num_symbols']}")
print(f"  num_languages: {config['num_languages']}")

## 6. ファインチューニング実行

Template B (シングルスピーカー) に準拠してファインチューニングを実行します。

| パラメータ | 値 | 理由 |
|-----------|-----|------|
| `--base_lr` | 2e-5 | catastrophic forgetting 防止 (事前学習の 1/10) |
| `--batch-size` | 4 | 小データセット向け |
| `--max_epochs` | 500 | 十分な gradient steps 確保 |
| `--freeze-dp` | auto | `--resume-from-multispeaker-checkpoint` で自動有効化 |

In [ ]:
# === 学習パラメータ ===
OUTPUT_DIR = f"/content/output-{SPEAKER_NAME}-finetune-6lang"
MAX_EPOCHS = 500
BATCH_SIZE = 4
LEARNING_RATE = 2e-5
CHECKPOINT_EPOCHS = 50
SAMPLES_PER_SPEAKER = 4

# WandB (オプション)
WANDB_API_KEY = ""  # WandB API キーを設定する場合はここに入力

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print("WandB logging enabled")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WandB logging disabled")

In [ ]:
# === ファインチューニング実行 ===
!cd /content/piper-plus && uv run python -m piper_train \
    --dataset-dir {DATASET_DIR} \
    --prosody-dim 16 \
    --accelerator gpu --devices 1 --precision 32-true \
    --max_epochs {MAX_EPOCHS} --batch-size {BATCH_SIZE} --samples-per-speaker {SAMPLES_PER_SPEAKER} \
    --checkpoint-epochs {CHECKPOINT_EPOCHS} --quality medium \
    --base_lr {LEARNING_RATE} --disable_auto_lr_scaling \
    --ema-decay 0.9995 \
    --max-phoneme-ids 400 \
    --no-wavlm \
    --val-every-n-epochs 50 \
    --audio-log-epochs 50 \
    --resume-from-multispeaker-checkpoint {BASE_CHECKPOINT} \
    --default_root_dir {OUTPUT_DIR}

## 7. ONNX エクスポート

In [ ]:
import glob
import re

# 最新のチェックポイントを検索 (epoch 番号でソート)
ckpt_pattern = f"{OUTPUT_DIR}/lightning_logs/version_*/checkpoints/epoch=*.ckpt"
checkpoints = glob.glob(ckpt_pattern)

def extract_epoch(path):
    """Extract epoch number from checkpoint filename for correct sorting."""
    m = re.search(r"epoch=(\d+)", path)
    return int(m.group(1)) if m else -1

checkpoints = sorted(checkpoints, key=extract_epoch)

if checkpoints:
    FINAL_CHECKPOINT = checkpoints[-1]
    print(f"Latest checkpoint: {FINAL_CHECKPOINT}")
else:
    print("No checkpoints found! Training may not have completed.")
    FINAL_CHECKPOINT = None

In [ ]:
ONNX_OUTPUT = f"{OUTPUT_DIR}/{SPEAKER_NAME}-6lang.onnx"

if FINAL_CHECKPOINT:
    !cd /content/piper-plus && CUDA_VISIBLE_DEVICES="" uv run python -m piper_train.export_onnx \
        {FINAL_CHECKPOINT} \
        {ONNX_OUTPUT}
    
    if os.path.exists(ONNX_OUTPUT):
        print(f"\nONNX model exported: {ONNX_OUTPUT}")
        print(f"Size: {os.path.getsize(ONNX_OUTPUT) / 1024**2:.1f} MB")
        
        import shutil
        onnx_config = ONNX_OUTPUT.replace(".onnx", ".onnx.json")
        shutil.copy2(config_path, onnx_config)
        print(f"Config copied: {onnx_config}")

## 8. 推論テスト

In [ ]:
# 言語に応じたテストテキスト
_inference_tests = {
    "ja": [
        "こんにちは、今日はとても良い天気ですね。",
        "吾輩は猫である。名前はまだ無い。",
    ],
    "en": [
        "Hello, how are you today?",
        "The quick brown fox jumps over the lazy dog.",
    ],
    "zh": [
        "你好，今天天气很好。",
        "欢迎使用语音合成系统。",
    ],
    "es": [
        "Hola, ¿cómo estás hoy?",
        "El rápido zorro marrón salta sobre el perro perezoso.",
    ],
    "fr": [
        "Bonjour, comment allez-vous aujourd'hui?",
        "Le renard brun rapide saute par-dessus le chien paresseux.",
    ],
    "pt": [
        "Olá, como você está hoje?",
        "A rápida raposa marrom pula sobre o cachorro preguiçoso.",
    ],
}

test_texts = [(t, PRIMARY_LANGUAGE) for t in _inference_tests.get(PRIMARY_LANGUAGE, _inference_tests["en"])]

if os.path.exists(ONNX_OUTPUT):
    for i, (text, lang) in enumerate(test_texts):
        test_output_dir = f"/content/test_output_{i}_{lang}"
        os.makedirs(test_output_dir, exist_ok=True)
        !cd /content/piper-plus && CUDA_VISIBLE_DEVICES="" uv run python -m piper_train.infer_onnx \
            --model {ONNX_OUTPUT} \
            --config {config_path} \
            --output-dir {test_output_dir} \
            --text "{text}" \
            --language {LANGUAGE_KEY} --speaker-id 0 --noise-scale 0.667
        print(f"Generated: {text[:40]}...")
else:
    print("ONNX model not found. Export first.")

In [ ]:
# Colab で音声を再生
from IPython.display import Audio, display

for i, (text, lang) in enumerate(test_texts):
    wav_path = f"/content/test_output_{i}_{lang}/output.wav"
    if os.path.exists(wav_path):
        print(f"\n[{lang}] {text[:40]}...")
        display(Audio(wav_path))
    else:
        print(f"\nNot found: {wav_path}")

## 9. モデル保存 (Google Drive)

学習済みモデルを Google Drive に保存します。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil

SAVE_DIR = f"/content/drive/MyDrive/piper-plus-models/{SPEAKER_NAME}-6lang"
os.makedirs(SAVE_DIR, exist_ok=True)

if os.path.exists(ONNX_OUTPUT):
    shutil.copy2(ONNX_OUTPUT, SAVE_DIR)
    shutil.copy2(ONNX_OUTPUT.replace(".onnx", ".onnx.json"), SAVE_DIR)
    print(f"Model saved to: {SAVE_DIR}")

if FINAL_CHECKPOINT:
    shutil.copy2(FINAL_CHECKPOINT, SAVE_DIR)
    print(f"Checkpoint saved to: {SAVE_DIR}")

!ls -lh {SAVE_DIR}

## 10. クリーンアップ

不要なキャッシュファイルを削除してディスク容量を確保します。

In [ ]:
# キャッシュディレクトリのサイズ確認
!du -sh {CACHE_DIR} 2>/dev/null
!df -h /content

# 必要に応じてキャッシュを削除
# !rm -rf {CACHE_DIR}